# Interview AI Studio: Natural Silence Removal and Neural Speech Denoising (Kaggle Edition)

![GPU Acceleration](https://img.shields.io/badge/Hardware-NVIDIA%20GPU%20NVENC-green)
![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-blue)
![License](https://img.shields.io/badge/License-MIT-purple)

Automate video post-production for interviews, podcasts, lectures, and talking-head content directly on Kaggle:
- Silero VAD v5: Deep-learning voice activity detection with configurable padding and pause bridging.
- DeepFilterNet 3 / Resemble Enhance: State-of-the-art neural speech noise suppression.
- Loudness Normalization: EBU R128 broadcast standard (-14.0 LUFS) with true-peak limiter.
- Hardware Accelerated NVENC: Fast GPU cutting and re-encoding with CPU libx264 fallback.
- NLE Timeline Export: Final Cut Pro 7 XML (.xml) and CMX 3600 EDL (.edl) for Adobe Premiere Pro and DaVinci Resolve.

Kaggle Setup Note: Ensure Internet is enabled in the notebook settings panel on the right (Settings -> Internet -> Internet On) and an accelerator is attached (GPU P100 or T4 x2).


In [ ]:
# Step 1: Environment Setup and Hardware Check
import os
import sys
import shutil
import torch

print("=" * 65)
print("Kaggle Environment and Hardware Verification:")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"[GPU] Runtime Active: {device_name} ({vram_gb:.1f} GB VRAM)")
    print(f"      CUDA Driver / Runtime Version: {torch.version.cuda}")
else:
    print("[WARN] No GPU detected! Running on CPU mode.")
    print("       Tip: In Kaggle right panel, select Settings -> Accelerator -> GPU P100/T4")
print("=" * 65)

# Fetch pipeline repository if running in fresh Kaggle session
if not os.path.exists("interview_processor.py"):
    print("Fetching Interview Studio pipeline repository...")
    !git clone -q https://github.com/RinmeSTD/DNColab.git _repo && cp -r _repo/* . && rm -rf _repo

# Ensure Cargo/Rust toolchain is available for DeepFilterNet compilation
cargo_bin = os.path.expanduser("~/.cargo/bin")
if not shutil.which("cargo") and not os.path.exists(os.path.join(cargo_bin, "cargo")):
    print("Setting up Rust toolchain for DeepFilterNet neural audio engine...")
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y > /dev/null 2>&1

os.environ["PATH"] = f"{cargo_bin}:{os.environ.get('PATH', '')}"

print("Installing required dependencies with uv (fast package manager)...")
!export PATH="$HOME/.cargo/bin:$PATH" && pip install -q uv && uv pip install --system -q soundfile scipy pyloudnorm tqdm deepfilternet ipywidgets || export PATH="$HOME/.cargo/bin:$PATH" && pip install -q soundfile scipy pyloudnorm tqdm deepfilternet ipywidgets

print("\nEnvironment ready for processing on Kaggle!")


In [ ]:
# Step 2: Storage and Dataset Discovery
import os
import glob

SUPPORTED_VIDEO_EXTS = (".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v")
KAGGLE_INPUT_ROOT = "/kaggle/input"
KAGGLE_WORKING_INPUT = "/kaggle/working/inputs"
KAGGLE_WORKING_OUTPUT = "/kaggle/working/outputs"

# Search for video files inside /kaggle/input dataset attachments
discovered_input_dir = None
if os.path.isdir(KAGGLE_INPUT_ROOT):
    for root, dirs, files in os.walk(KAGGLE_INPUT_ROOT):
        if any(f.lower().endswith(SUPPORTED_VIDEO_EXTS) for f in files):
            discovered_input_dir = root
            break

if discovered_input_dir:
    active_input_dir = discovered_input_dir
    print(f"[DATASET] Auto-detected input video dataset at: {active_input_dir}")
elif os.path.isdir("/kaggle/working"):
    active_input_dir = KAGGLE_WORKING_INPUT
    print(f"[WORKSPACE] Using Kaggle working inputs folder: {active_input_dir}")
else:
    active_input_dir = "./inputs"
    print(f"[LOCAL] Using local workspace folder: {active_input_dir}")

active_output_dir = KAGGLE_WORKING_OUTPUT if os.path.isdir("/kaggle/working") else "./outputs"

os.makedirs(active_input_dir, exist_ok=True)
os.makedirs(active_output_dir, exist_ok=True)

print(f"Active Input Directory  : {os.path.abspath(active_input_dir)}")
print(f"Active Output Directory : {os.path.abspath(active_output_dir)}")
print("\nTip: You can attach a Kaggle Dataset containing video files or upload to the input folder.")


In [ ]:
# Step 3: Interactive ipywidgets Configuration GUI Dashboard
import ipywidgets as widgets
from IPython.display import display

# Text inputs for directory paths
widget_input_dir = widgets.Text(
    value=active_input_dir,
    description="Input Dir:",
    layout=widgets.Layout(width="75%")
)
widget_output_dir = widgets.Text(
    value=active_output_dir,
    description="Output Dir:",
    layout=widgets.Layout(width="75%")
)

# Denoise engine dropdown
widget_denoise_engine = widgets.Dropdown(
    options=["DeepFilterNet3", "ResembleEnhance", "None"],
    value="DeepFilterNet3",
    description="AI Denoise:",
    layout=widgets.Layout(width="50%")
)

# Silence cut and padding sliders
widget_min_silence = widgets.FloatSlider(
    value=0.8,
    min=0.2,
    max=2.0,
    step=0.05,
    description="Min Silence (s):",
    continuous_update=False,
    layout=widgets.Layout(width="65%")
)
widget_padding = widgets.FloatSlider(
    value=0.25,
    min=0.05,
    max=0.5,
    step=0.01,
    description="Padding (s):",
    continuous_update=False,
    layout=widgets.Layout(width="65%")
)
widget_crossfade = widgets.IntSlider(
    value=30,
    min=10,
    max=100,
    step=5,
    description="Crossfade (ms):",
    continuous_update=False,
    layout=widgets.Layout(width="65%")
)

# Options checkboxes
widget_normalize = widgets.Checkbox(
    value=True,
    description="Normalize Audio (-14.0 LUFS EBU R128)",
    indent=False
)
widget_export_timeline = widgets.Checkbox(
    value=True,
    description="Export NLE Timelines (FCP7 XML and CMX 3600 EDL)",
    indent=False
)
widget_use_gpu = widgets.Checkbox(
    value=True,
    description="GPU Acceleration (NVIDIA NVENC with CPU fallback)",
    indent=False
)
widget_overwrite = widgets.Checkbox(
    value=False,
    description="Overwrite Existing Rendered Outputs",
    indent=False
)

# Render interactive dashboard
dashboard = widgets.VBox([
    widgets.HTML("<h3>Interview AI Studio Configuration Dashboard</h3>"),
    widgets.VBox([widget_input_dir, widget_output_dir]),
    widgets.HBox([widget_denoise_engine]),
    widgets.VBox([widget_min_silence, widget_padding, widget_crossfade]),
    widgets.VBox([widget_normalize, widget_export_timeline, widget_use_gpu, widget_overwrite])
])
display(dashboard)


In [ ]:
# Step 4: Run Batch Processing Queue
import os
import sys
import time
import json
from interview_processor import process_batch

# Read active values from widgets with fallback defaults
active_input = widget_input_dir.value if "widget_input_dir" in globals() else active_input_dir
active_output = widget_output_dir.value if "widget_output_dir" in globals() else active_output_dir

pipeline_config = {
    "denoise_engine": widget_denoise_engine.value if "widget_denoise_engine" in globals() else "DeepFilterNet3",
    "min_silence_sec": float(widget_min_silence.value if "widget_min_silence" in globals() else 0.8),
    "padding_sec": float(widget_padding.value if "widget_padding" in globals() else 0.25),
    "crossfade_ms": int(widget_crossfade.value if "widget_crossfade" in globals() else 30),
    "normalize_audio": bool(widget_normalize.value if "widget_normalize" in globals() else True),
    "export_timeline": bool(widget_export_timeline.value if "widget_export_timeline" in globals() else True),
    "use_gpu": bool(widget_use_gpu.value if "widget_use_gpu" in globals() else True),
    "overwrite": bool(widget_overwrite.value if "widget_overwrite" in globals() else False),
}

os.makedirs(active_output, exist_ok=True)

print("=" * 75)
print("Commencing batch processing queue on Kaggle...")
print(f"   Input Path : {os.path.abspath(active_input)}")
print(f"   Output Path: {os.path.abspath(active_output)}")
print(f"   Engine     : {pipeline_config['denoise_engine']}")
print(f"   Min Silence: {pipeline_config['min_silence_sec']}s | Padding: {pipeline_config['padding_sec']*1000:.0f}ms")
print("=" * 75 + "\n")

batch_start_time = time.time()
batch_results = process_batch(active_input, active_output, pipeline_config)
total_elapsed_time = time.time() - batch_start_time

# Calculate summary metrics
successful_cuts = [r for r in batch_results if r.get("status") == "success"]
skipped_files = [r for r in batch_results if r.get("status") == "skipped"]
failed_files = [r for r in batch_results if r.get("status") == "failed"]

total_orig_duration = sum(r.get("stats", {}).get("original_duration", 0.0) for r in successful_cuts)
total_kept_duration = sum(r.get("stats", {}).get("kept_duration", 0.0) for r in successful_cuts)
total_silence_cut = sum(r.get("stats", {}).get("silence_removed", 0.0) for r in successful_cuts)
overall_silence_pct = (total_silence_cut / total_orig_duration * 100.0) if total_orig_duration > 0 else 0.0
speed_factor = (total_orig_duration / total_elapsed_time) if total_elapsed_time > 0 else 0.0

print("\n" + "=" * 75)
print("BATCH EXECUTION SUMMARY REPORT")
print("=" * 75)
print(f"* Total Videos Queued : {len(batch_results)}")
print(f"* Successfully Cut    : {len(successful_cuts)}")
print(f"* Skipped (Unchanged) : {len(skipped_files)}")
print(f"* Failed / Errors     : {len(failed_files)}")
print("-" * 75)
print(f"* Original Duration   : {total_orig_duration:.2f}s ({total_orig_duration / 60:.2f} minutes)")
print(f"* Clean Cut Duration  : {total_kept_duration:.2f}s ({total_kept_duration / 60:.2f} minutes)")
print(f"* Duration Saved      : {total_silence_cut:.2f}s ({total_silence_cut / 60:.2f} minutes)")
print(f"* Silence Cut Ratio   : {overall_silence_pct:.1f}%")
print(f"* Total Execution Time: {total_elapsed_time:.2f}s")
print(f"* Batch Speed Factor  : {speed_factor:.2f}x Realtime speed")
print("=" * 75)

if successful_cuts:
    print("\nPer-File Breakdown:")
    for item in successful_cuts:
        base = os.path.basename(item.get("file", ""))
        s = item.get("stats", {})
        t = item.get("processing_time_sec", 0.0)
        print(f"  * {base:<28} | {s.get('original_duration', 0):.1f}s -> {s.get('kept_duration', 0):.1f}s (-{s.get('silence_percentage', 0):.1f}%) | {t:.1f}s processing")


In [ ]:
# Step 5: Preview Results and Download Zip Archive
import os
import glob
import shutil
from IPython.display import display, Video, FileLink, HTML

active_output = widget_output_dir.value if "widget_output_dir" in globals() else active_output_dir

# 1. Preview first rendered clean-cut video
output_videos = sorted(glob.glob(os.path.join(active_output, "*_clean_cut.mp4")))
if output_videos:
    preview_target = output_videos[0]
    print(f"Interactive Player Preview: {os.path.basename(preview_target)}")
    try:
        display(Video(preview_target, embed=True, width=640, height=360))
    except Exception as err:
        print(f"Player embed info: {err}")
else:
    print("[INFO] No rendered video files found in output directory.")

# 2. Package outputs into a single .zip archive in /kaggle/working
archive_basename = "Interview_AI_Studio_Outputs"
archive_dest = os.path.join("/kaggle/working", archive_basename) if os.path.exists("/kaggle/working") else os.path.join(active_output, "..", archive_basename)
zip_file_path = shutil.make_archive(archive_dest, "zip", active_output)
archive_size_mb = os.path.getsize(zip_file_path) / (1024 * 1024)
print(f"\nResults archive bundled successfully: {zip_file_path} ({archive_size_mb:.2f} MB)")

# 3. Display downloadable file link in Kaggle
try:
    display(FileLink(zip_file_path, result_html_prefix="Click here to download outputs archive: "))
except Exception:
    pass

print("\nTip: In Kaggle, all generated files and ZIP archives in /kaggle/working/ are available in the 'Output' tab in the right sidebar for instant download.")
